# Setup

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

## IMPORTS
import os

import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from shapely.geometry import Point, box

import src.data_filepaths as data_filepaths
import src.utils.loaders as loaders
import src.utils.vis.colors_utils as colors_utils
from src.utils.df_tools import map_values_b_to_bp
from src.utils.loaders import process_loaded_building_df
from src.utils.vis.map import CategoricalColorMap, MapExplorer
from src.utils.vis.map_special import build_map_point_numeric_and_category

# configs
sns.set_theme()

# Boundaries

### Schema Description

**Planning Regions**:

* `OBJECTID`
* `REGION_N`: name
* `REGION_C`: code
* `INC_CRC`, `FMEL_UPD_D`
* `SHAPE.AREA`, `SHAPE.LEN`, `geometry`

**Planning Areas**:

* `OBJECTID`
* `PLN_AREA_N`: name
* `PLN_AREA_C`: code
* `REGION_N`, `REGION_C`
* `INC_CRC`, `FMEL_UPD_D`
* `SHAPE.AREA`, `SHAPE.LEN`, `geometry`

**Subzones**:

* `OBJECTID`
* `SUBZONE_NO`
* `SUBZONE_N`: name
* `SUBZONE_C`: code
* `PLN_AREA_N`, `PLN_AREA_C`, `REGION_N`, `REGION_C`
* `INC_CRC`, `FMEL_UPD_D`
* `SHAPE.AREA`, `SHAPE.LEN`, `geometry`

In [2]:
# load them
pln_region_gdf = gpd.read_file(data_filepaths.BASE_PLN_REGION)
pln_area_gdf = gpd.read_file(data_filepaths.BASE_PLN_AREA)
subzone_gdf = gpd.read_file(data_filepaths.BASE_SUBZONE)

In [3]:
pln_region_gdf

,OBJECTID,REGION_N,REGION_C,INC_CRC,FMEL_UPD_D,SHAPE.AREA,SHAPE.LEN,geometry
0,1,CENTRAL REGION,CR,63C6FAD3D10B7D2E,20251229123457,1.364132e+08,129647.353464,"MULTIPOLYGON (((103.86123 1.22134, 103.8609 1...."
1,2,EAST REGION,ER,FFB1DA36DCF2CEF2,20251229123458,1.251725e+08,61947.837014,"MULTIPOLYGON (((104.08825 1.31344, 104.08828 1..."
2,3,WEST REGION,WR,13CE24061D199BFC,20251229123458,2.598394e+08,254921.899215,"MULTIPOLYGON (((103.74134 1.15997, 103.7413 1...."
3,4,NORTH REGION,NR,168D0901E5DE6ACB,20251229123458,1.394169e+08,87294.732067,"MULTIPOLYGON (((103.87242 1.43995, 103.87236 1..."
4,5,NORTH-EAST REGION,NER,FD78445FD6BC22A4,20251229123458,1.360618e+08,106714.379971,"MULTIPOLYGON (((104.0645 1.43547, 104.06452 1...."


In [4]:
pln_area_gdf.head()

,OBJECTID,PLN_AREA_N,PLN_AREA_C,CA_IND,REGION_N,REGION_C,INC_CRC,FMEL_UPD_D,SHAPE.AREA,SHAPE.LEN,geometry
0,28,BUKIT TIMAH,BT,N,CENTRAL REGION,CR,A3BFBAC3EBE4A93B,20251229123848,1.751829e+07,22652.993950,"POLYGON ((103.79644 1.34888, 103.79569 1.34917..."
1,29,CLEMENTI,CL,N,WEST REGION,WR,4D34F1FA6471066E,20251229123848,9.523534e+06,15044.652233,"POLYGON ((103.75867 1.29922, 103.76102 1.30103..."
2,30,JURONG EAST,JE,N,WEST REGION,WR,CB9A67C633D272C7,20251229123848,1.786953e+07,26938.090843,"POLYGON ((103.75728 1.29837, 103.75726 1.29842..."
3,31,NOVENA,NV,N,CENTRAL REGION,CR,163B19CA54A679AF,20251229123848,8.968012e+06,14804.900494,"POLYGON ((103.83599 1.34168, 103.83598 1.34168..."
4,32,QUEENSTOWN,QT,N,CENTRAL REGION,CR,6FEECCA10F1E3D3C,20251229123848,2.168363e+07,34363.162430,"POLYGON ((103.79831 1.26663, 103.79921 1.2682,..."


In [5]:
subzone_gdf

,OBJECTID,SUBZONE_NO,SUBZONE_N,SUBZONE_C,CA_IND,PLN_AREA_N,PLN_AREA_C,REGION_N,REGION_C,INC_CRC,FMEL_UPD_D,SHAPE.AREA,SHAPE.LEN,geometry
0,676,12,DEPOT ROAD,BMSZ12,N,BUKIT MERAH,BM,CENTRAL REGION,CR,C22DED671DE2A940,20191223152313,4.422975e+05,4281.667289,"POLYGON ((103.81454 1.28239, 103.81427 1.28253..."
1,677,2,BUKIT MERAH,BMSZ02,N,BUKIT MERAH,BM,CENTRAL REGION,CR,085EF219A5A1AEAD,20191223152313,4.117228e+05,3074.963234,"POLYGON ((103.82209 1.28049, 103.82209 1.28057..."
2,678,3,CHINATOWN,OTSZ03,Y,OUTRAM,OT,CENTRAL REGION,CR,EF2B9A91AF49E025,20191223152313,5.872227e+05,4297.599898,"POLYGON ((103.84375 1.28508, 103.84363 1.28518..."
3,679,4,PHILLIP,DTSZ04,Y,DOWNTOWN CORE,DT,CENTRAL REGION,CR,615D4EDDEF809F8E,20191223152313,3.943794e+04,871.554888,"POLYGON ((103.84962 1.28412, 103.84963 1.28414..."
4,680,5,RAFFLES PLACE,DTSZ05,Y,DOWNTOWN CORE,DT,CENTRAL REGION,CR,72107B11807074F4,20191223152313,1.887675e+05,1872.752161,"POLYGON ((103.85253 1.28617, 103.85236 1.28651..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
327,968,1,UPPER THOMSON,BSSZ01,N,BISHAN,BS,CENTRAL REGION,CR,716DA27F6666AB0C,20191223152313,3.849507e+06,10031.378828,"POLYGON ((103.84263 1.36323, 103.84297 1.36429..."
328,969,5,SHANGRI-LA,AMSZ05,N,ANG MO KIO,AM,NORTH-EAST REGION,NER,9FA8567B39D8D9D7,20191223152313,6.879137e+05,4019.215248,"POLYGON ((103.83537 1.37093, 103.83496 1.37097..."
329,970,4,TOWNSVILLE,AMSZ04,N,ANG MO KIO,AM,NORTH-EAST REGION,NER,F5558D84EBC2AAA7,20191223152313,5.463938e+05,3313.322844,"POLYGON ((103.84814 1.36879, 103.84795 1.3688,..."
330,971,2,MARYMOUNT,BSSZ02,N,BISHAN,BS,CENTRAL REGION,CR,154AA8659ADDE6D7,20191223152313,1.964145e+06,6383.484708,"POLYGON ((103.84652 1.34521, 103.84709 1.34778..."


## Load More Cleanly

In [6]:
def load_boundary_pln_region(fpath: str = data_filepaths.BASE_PLN_REGION, crs=3414):
    """
    Initial Schema:
    * `OBJECTID`
    * `REGION_N`: name
    * `REGION_C`: code
    * `INC_CRC`, `FMEL_UPD_D`
    * `SHAPE.AREA`, `SHAPE.LEN`, `geometry`

    Output Schema:
    * `pln_region_id` as index
    * `pln_region_name`
    * `area`, `geometry`

    """

    loaded_gdf = gpd.read_file(fpath)
    loaded_gdf = loaded_gdf.to_crs(crs)

    new_gdf = loaded_gdf[["REGION_N", "REGION_C", "SHAPE.AREA", "geometry"]]
    new_gdf = new_gdf.rename(
        columns={
            "REGION_N": "pln_region_name",
            "REGION_C": "pln_region_id",
            "SHAPE.AREA": "area",
        }
    )
    new_gdf = new_gdf.set_index("pln_region_id")

    return new_gdf


load_boundary_pln_region().head()

,pln_region_name,area,geometry
pln_region_id,,,
CR,CENTRAL REGION,1.364132e+08,"MULTIPOLYGON (((31106.551 22674.669, 31069.858..."
ER,EAST REGION,1.251725e+08,"MULTIPOLYGON (((56372.004 32860.809, 56374.439..."
WR,WEST REGION,2.598394e+08,"MULTIPOLYGON (((17763.386 15889.099, 17759.053..."
NR,NORTH REGION,1.394169e+08,"MULTIPOLYGON (((32351.432 46848.428, 32344.609..."
NER,NORTH-EAST REGION,1.360618e+08,"MULTIPOLYGON (((53726.907 46354.201, 53729.653..."


In [7]:
def load_boundary_pln_area(fpath: str = data_filepaths.BASE_PLN_AREA, crs=3414):
    """
    Initial Schema:
    * `OBJECTID`
    * `PLN_AREA_N`: name
    * `PLN_AREA_C`: code
    * `REGION_N`, `REGION_C`
    * `INC_CRC`, `FMEL_UPD_D`
    * `SHAPE.AREA`, `SHAPE.LEN`, `geometry`

    Output Schema:
    * `pln_area_id` as index
    * `pln_area_name`
    * `pln_region_id`
    * `pln_region_name`
    * `area`, `geometry`

    """

    loaded_gdf = gpd.read_file(fpath)
    loaded_gdf = loaded_gdf.to_crs(crs)

    new_gdf = loaded_gdf[
        ["PLN_AREA_N", "PLN_AREA_C", "REGION_N", "REGION_C", "SHAPE.AREA", "geometry"]
    ]
    new_gdf = new_gdf.rename(
        columns={
            "PLN_AREA_N": "pln_area_name",
            "PLN_AREA_C": "pln_area_id",
            "REGION_N": "pln_region_name",
            "REGION_C": "pln_region_id",
            "SHAPE.AREA": "area",
        }
    )
    new_gdf = new_gdf.set_index("pln_area_id")

    return new_gdf


load_boundary_pln_area().head()

,pln_area_name,pln_region_name,pln_region_id,area,geometry
pln_area_id,,,,,
BT,BUKIT TIMAH,CENTRAL REGION,CR,1.751829e+07,"POLYGON ((23895.572 36778.113, 23812.687 36810..."
CL,CLEMENTI,WEST REGION,WR,9.523534e+06,"POLYGON ((19691.762 31286.481, 19954.25 31486...."
JE,JURONG EAST,WEST REGION,WR,1.786953e+07,"POLYGON ((19537.709 31192.991, 19535.405 31198..."
NV,NOVENA,CENTRAL REGION,CR,8.968012e+06,"POLYGON ((28297.1 35982.083, 28295.925 35981.3..."
QT,QUEENSTOWN,CENTRAL REGION,CR,2.168363e+07,"POLYGON ((24104.224 27683.347, 24203.984 27856..."


In [8]:
def load_boundary_subzone(fpath: str = data_filepaths.BASE_SUBZONE, crs=3414):
    """
    Initial Schema:
    * `OBJECTID`
    * `SUBZONE_NO`
    * `SUBZONE_N`: name
    * `SUBZONE_C`: code
    * `PLN_AREA_N`, `PLN_AREA_C`, `REGION_N`, `REGION_C`
    * `INC_CRC`, `FMEL_UPD_D`
    * `SHAPE.AREA`, `SHAPE.LEN`, `geometry`

    Output Schema:
    * `subzone_id` as index
    * `subzone_name`
    * `pln_area_id`
    * `pln_area_name`
    * `pln_region_id`
    * `pln_region_name`
    * `area`, `geometry`

    """

    loaded_gdf = gpd.read_file(fpath)
    loaded_gdf = loaded_gdf.to_crs(crs)

    new_gdf = loaded_gdf[
        [
            "SUBZONE_N",
            "SUBZONE_C",
            "PLN_AREA_N",
            "PLN_AREA_C",
            "REGION_N",
            "REGION_C",
            "SHAPE.AREA",
            "geometry",
        ]
    ]
    new_gdf = new_gdf.rename(
        columns={
            "SUBZONE_N": "subzone_name",
            "SUBZONE_C": "subzone_id",
            "PLN_AREA_N": "pln_area_name",
            "PLN_AREA_C": "pln_area_id",
            "REGION_N": "pln_region_name",
            "REGION_C": "pln_region_id",
            "SHAPE.AREA": "area",
        }
    )
    new_gdf = new_gdf.set_index("subzone_id")

    return new_gdf


load_boundary_subzone().head()

,subzone_name,pln_area_name,pln_area_id,pln_region_name,pln_region_id,area,geometry
subzone_id,,,,,,,
BMSZ12,DEPOT ROAD,BUKIT MERAH,BM,CENTRAL REGION,CR,442297.509556,"POLYGON ((25910.344 29425.34, 25880.344 29441...."
BMSZ02,BUKIT MERAH,BUKIT MERAH,BM,CENTRAL REGION,CR,411722.830243,"POLYGON ((26750.092 29216.098, 26750.091 29224..."
OTSZ03,CHINATOWN,OUTRAM,OT,CENTRAL REGION,CR,587222.679344,"POLYGON ((29161.201 29723.071, 29147.201 29734..."
DTSZ04,PHILLIP,DOWNTOWN CORE,DT,CENTRAL REGION,CR,39437.935270,"POLYGON ((29814.107 29616.894, 29814.868 29619..."
DTSZ05,RAFFLES PLACE,DOWNTOWN CORE,DT,CENTRAL REGION,CR,188767.489706,"POLYGON ((30137.768 29843.194, 30118.735 29880..."
